# Retrieval Benchmark Evaluation

Evaluate 7 Colette retrieval methods against a **119-question** ground-truth QA dataset
spanning 5 original VTQ PDFs plus 2 new **Option B** PDFs with chart-only data.
Computes retrieval metrics (Hit@K, P@K, R@K, MRR, NDCG@K) and answer-quality metrics
(Exact Match, Token-F1, ROUGE-1/2/L, Contains, Semantic Similarity), then produces
comparison charts across question types.

**Sections:**
1. Setup & Configuration
2. Service Configuration & Indexing
3. Load Ground Truth
4. Run All Queries
5. Retrieval Metrics
6. Answer Quality Metrics
7. Summary Tables
8. Visualizations
9. Per-Question Deep Dive
10. Key Findings


## 1. Setup & Configuration

In [ ]:
import os, sys, json, re, time, warnings, math
from pathlib import Path
from io import BytesIO
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import defaultdict, Counter
from PIL import Image
warnings.filterwarnings("ignore")

colette_root  = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(colette_root, "src"))
BENCHMARK_DIR = Path(os.getcwd()) / "benchmark"

# ── Must match setup_eval_service.ipynb ─────────────────────────────────────
app_name    = "eval_benchmark"
app_dir     = os.path.join(colette_root, "app_eval_benchmark")
TOP_K       = 4
# GPU for LLM in *this* notebook's process.
# GPU 1 is reserved for setup_eval_service.ipynb; GPU 2 is free when both run together.
# Change to [1] if the setup notebook kernel is not running.
EVAL_LLM_GPU_IDS = [2]
# ────────────────────────────────────────────────────────────────────────────

from colette.jsonapi import JSONApi
from colette.apidata import APIData

colette_api = JSONApi()
print(f"Connected — app: {app_name}  data: {BENCHMARK_DIR}")
print(f"LLM will load on GPU(s): {EVAL_LLM_GPU_IDS}")
print("Service must already be indexed (see setup_eval_service.ipynb).")


In [ ]:
# ── Source-aware matching: map UUID key prefixes → document basenames ────────
# Built from hindex JSON files at eval time so no re-run of queries is needed.
# Keys in image_hits are UUID-prefixed (e.g. "ad3f9e57-..._0002_crop_0000").
# The first 36 characters are the document UUID; the rest is page/crop info.

HINDEX_DIR = Path(app_dir) / "mm_index" / "hindex"
UUID_TO_DOC: dict = {}   # "ad3f9e57-0bb0-..." → "VTQ-Q1-2025-Architecture.pdf"

if HINDEX_DIR.exists():
    for jf in sorted(HINDEX_DIR.glob("*.json")):
        try:
            d = json.loads(jf.read_text())
            if d.get("page_records"):
                uuid_prefix = d["page_records"][0]["page_key"][:36]
                UUID_TO_DOC[uuid_prefix] = os.path.basename(d["doc_name"])
        except Exception:
            pass

print(f"Source map: {len(UUID_TO_DOC)} documents")
for u, n in sorted(UUID_TO_DOC.items(), key=lambda x: x[1]):
    print(f"  {u[:8]}…  →  {n}")


In [ ]:
METHODS = [
    {
        "name": "embedding", "label": "Embedding", "color": "#2196F3",
        "config": {"retrieval_mode": "embedding_retrieval"},
    },
    {
        "name": "text_search", "label": "Text Search", "color": "#FF9800",
        "config": {"retrieval_mode": "text_search_retrieval"},
    },
    {
        "name": "hybrid", "label": "Hybrid", "color": "#4CAF50",
        "config": {"retrieval_mode": "hybrid"},
    },
    {
        "name": "vision_hier", "label": "hindex only", "color": "#9C27B0",
        "config": {"retrieval_mode": "vision_hierarchical"},
    },
    {
        "name": "embed_hindex", "label": "Embed+hindex", "color": "#00BCD4",
        "config": {"retrieval_mode": "embedding_retrieval", "hindex": {"enable_hindex": True}},
    },
    {
        "name": "text_hindex", "label": "Text+hindex", "color": "#795548",
        "config": {"retrieval_mode": "text_search_retrieval", "hindex": {"enable_hindex": True}},
    },
    {
        "name": "hybrid_hindex", "label": "Hybrid+hindex", "color": "#F44336",
        "config": {"retrieval_mode": "hybrid", "hindex": {"enable_hindex": True}},
    },
]

METHOD_LABELS = [m["label"] for m in METHODS]
METHOD_COLORS = [m["color"] for m in METHODS]
print(f"Configured {len(METHODS)} retrieval methods.")


### Table 0 — Method Definitions

| Method | Image source | Text source | hindex | Notes |
|---|---|---|:---:|---|
| **Embedding** | Dense vector (GME-Qwen2-VL-2B) — crops | — | ✗ | Pure semantic similarity over page crops |
| **Text Search** | — | BM25 full-text | ✗ | Keyword precision; no image context |
| **Hybrid** | Dense vector — crops | BM25 | ✗ | Semantic + keyword; union, vector-ranked first |
| **hindex** | VLM page-selection — full pages + crops | — | ✓ | Hierarchical nav; position-based distances (0.05–0.20) |
| **Embed+hindex** | hindex (lower dist) + dense vector — sorted merge | — | ✓ | hindex fills top-k; embedding supplements |
| **Text+hindex** | VLM page-selection — full pages + crops | BM25 | ✓ | hindex for images, BM25 for text |
| **Hybrid+hindex** | hindex + dense vector — sorted merge | BM25 | ✓ | Maximum multi-signal coverage |

> **Distance ordering (Embed+hindex, Hybrid+hindex):** hindex distances (0.05–0.20) are smaller than embedding cosine distances (0.4–0.8). After the sorted merge, hindex pages rank first in the image context seen by the LLM.


## 2. Connect to Service

> **Pre-requisite:** run `setup_eval_service.ipynb` first to index the benchmark PDFs.
> This notebook only queries the already-running service.


In [ ]:
# Ensure service is loaded — load from saved config if not already in memory
_cfg_path = Path(app_dir) / "config.json"

try:
    info = colette_api.service_info()
    running = list(info.info.services) if info.info else []
except Exception:
    running = []

if app_name in running:
    print(f"Service '{app_name}' is already loaded.")
else:
    if not _cfg_path.exists():
        raise RuntimeError(
            f"No saved config at {_cfg_path}.\n"
            "Run setup_eval_service.ipynb first to create the service and index."
        )
    saved_cfg = json.loads(_cfg_path.read_text())
    # Override LLM GPU — GPU 1 may be occupied by the setup notebook kernel.
    saved_cfg["parameters"]["llm"]["gpu_ids"] = EVAL_LLM_GPU_IDS
    print(f"Loading service '{app_name}' from saved config (LLM → GPU {EVAL_LLM_GPU_IDS})…")
    colette_api.service_create(app_name, APIData(**saved_cfg))
    print(f"Service '{app_name}' loaded successfully.")


## 3. Load Ground Truth

In [ ]:
QA_ITEMS = []
with open(BENCHMARK_DIR / "qa_dataset.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            QA_ITEMS.append(json.loads(line))

print(f"Loaded {len(QA_ITEMS)} QA items")

# Distribution table
from collections import Counter
type_counts = Counter(q["question_type"] for q in QA_ITEMS)
diff_counts = Counter(q["difficulty"] for q in QA_ITEMS)

df_dist = pd.DataFrame([
    {"question_type": qt, "easy": sum(1 for q in QA_ITEMS if q["question_type"] == qt and q["difficulty"] == "easy"),
     "medium": sum(1 for q in QA_ITEMS if q["question_type"] == qt and q["difficulty"] == "medium"),
     "hard": sum(1 for q in QA_ITEMS if q["question_type"] == qt and q["difficulty"] == "hard"),
     "total": cnt}
    for qt, cnt in sorted(type_counts.items())
])
print("\nDistribution by question type and difficulty:")
display(df_dist)
print(f"\nDifficulty: {dict(diff_counts)}")


## 4. Run All Queries

In [ ]:
def _decode_image(data_uri: str):
    try:
        b64 = re.sub(r"^data:image/.+;base64,", "", data_uri)
        return Image.open(BytesIO(base64.b64decode(b64)))
    except Exception:
        return None


def run_query(method: dict, question: str, top_k: int = 4) -> dict:
    rag_params = dict(method["config"])
    rag_params["top_k"] = top_k
    payload = {"parameters": {"input": {"message": question, "rag": rag_params}}}
    t0 = time.time()
    response = colette_api.service_predict(app_name, APIData(**payload))
    elapsed = round(time.time() - t0, 2)

    # response.sources is None when the service errored or returned no context
    sources = response.sources or {}

    image_hits = []
    for item in sources.get("context", []):
        image_hits.append({
            "key":              item.get("key", ""),
            "page_number":      item.get("page_number"),
            "distance":         item.get("distance", float("nan")),
            "crop_label":       item.get("crop_label", ""),
            "retrieval_source": item.get("retrieval_source", "embedding"),
            "summary":          item.get("summary", ""),
        })

    text_hits = [
        {
            "source":      h.get("source", ""),
            "page_number": h.get("page_number"),
            "score":       h.get("score", float("nan")),
            "content":     h.get("content", ""),
            "crop_label":  h.get("crop_label", ""),
        }
        for h in sources.get("text_context", [])
    ]

    return {
        "method":     method["name"],
        "question":   question,
        "answer":     response.output or "",
        "latency_s":  elapsed,
        "image_hits": image_hits,
        "text_hits":  text_hits,
    }


In [ ]:
# ── Cache invalidation ────────────────────────────────────────────────────────
# After a code change that alters retrieval behaviour, add the affected method
# names here to force a fresh run. Empty the set once re-run is complete.
#
# Code changes that affect cached results:
#   • vision_hierarchical now runs hindex ONLY (no embedding retrieval)  → vision_hier
#   • _merge_hindex_results now sorts by distance ascending              → embed_hindex, hybrid_hindex
#   • balanced slot allocation (ceil(top_k/2) per source)               → embed_hindex, hybrid_hindex
# Re-run completed 2026-07-22; all 833 entries verified correct.
FORCE_RERUN_METHODS: frozenset = frozenset()
# ─────────────────────────────────────────────────────────────────────────────

RESULTS_FILE = Path(app_dir) / "eval_results.json"
ALL_RESULTS: dict = {}

if RESULTS_FILE.exists():
    raw = json.loads(RESULTS_FILE.read_text())
    ALL_RESULTS = {
        tuple(k.split("|", 1)): v
        for k, v in raw.items()
        if k.split("|")[0] not in FORCE_RERUN_METHODS
    }
    n_dropped = len(raw) - len(ALL_RESULTS)
    msg = f" (dropped {n_dropped} stale entries for {set(FORCE_RERUN_METHODS)})" if n_dropped else ""
    print(f"Loaded {len(ALL_RESULTS)} cached results from {RESULTS_FILE.name}{msg}")

missing = [
    (m, item)
    for item in QA_ITEMS
    for m in METHODS
    if (m["name"], item["id"]) not in ALL_RESULTS
]

if missing:
    total = len(QA_ITEMS) * len(METHODS)
    print(f"\n{len(missing)}/{total} queries not yet cached — running them now…")
    for done, (m, item) in enumerate(missing, 1):
        qid = item["id"]
        q   = item["question"]
        print(f"[{done}/{len(missing)}] {m['label']:<14} {qid}  {q[:55]}…", end=" ", flush=True)
        try:
            r = run_query(m, q, top_k=TOP_K)
            ALL_RESULTS[(m["name"], qid)] = r
            print(f"✓ {r['latency_s']:.1f}s  img={len(r['image_hits'])}  txt={len(r['text_hits'])}")
        except Exception as exc:
            print(f"ERROR: {exc}")

    serializable = {f"{mn}|{qid}": r for (mn, qid), r in ALL_RESULTS.items()}
    RESULTS_FILE.write_text(json.dumps(serializable, indent=2, default=str))
    print(f"\nResults saved → {RESULTS_FILE}")
else:
    print("All queries already cached — nothing to run.")
    print("To force a re-run: add method name(s) to FORCE_RERUN_METHODS above.")

print(f"\nTotal results in memory: {len(ALL_RESULTS)}")

## 5. Retrieval Metrics

In [ ]:
# ── Source-aware retrieval helpers ────────────────────────────────────────────
# All retrieved-page lists are (doc_basename, page_number) tuples.
# Source is extracted from image-hit keys (UUID prefix → UUID_TO_DOC)
# and from text-hit source paths (basename extraction).
# This eliminates false positives when the same page number appears in
# multiple documents (critical for cross-document evaluation).

def _doc_from_key(key: str) -> str:
    """Extract document basename from an image-hit key (first 36 chars = UUID)."""
    return UUID_TO_DOC.get(key[:36], "") if len(key) >= 36 else ""


def _doc_from_path(path: str) -> str:
    """Extract document basename from a text-hit source path."""
    return os.path.basename(path) if path else ""


def _dedup_hits(hits_list: list, is_text: bool = False) -> list:
    """Return (doc_basename, page_number) pairs in retrieval order, deduped."""
    seen, result = set(), []
    for h in hits_list:
        p = h.get("page_number")
        if p is None:
            continue
        src = (_doc_from_path(h.get("source", "")) if is_text
               else _doc_from_key(h.get("key", "")))
        pair = (src, p)
        if pair not in seen:
            seen.add(pair)
            result.append(pair)
    return result


def get_retrieved_pages(result: dict, method_name: str, top_k: int = TOP_K) -> list:
    """Return ordered (doc_basename, page_number) pairs for a cached result.

    Single-channel methods (embedding, text_search, vision_hier, embed_hindex)
    get top_k items from their one channel.

    Dual-channel methods (hybrid, text_hindex, hybrid_hindex) get top_k from
    EACH channel — up to 2*top_k items total — because that is exactly what
    the LLM receives as context. Evaluating only a 2+2 split would penalise
    these methods by discarding half their retrieved evidence.

    Deduplication is applied across channels so a page that appears in both
    image and text results is only counted once (image rank wins).
    """
    image_pages = _dedup_hits(result.get("image_hits", []), is_text=False)
    text_pages  = _dedup_hits(result.get("text_hits",  []), is_text=True)

    if method_name == "text_search":
        return text_pages[:top_k]

    elif method_name in ("embedding", "vision_hier", "embed_hindex"):
        return image_pages[:top_k]

    else:
        # Dual-channel: full top_k from each channel (image first, matching LLM context order)
        img_slot = image_pages[:top_k]
        txt_slot = text_pages[:top_k]
        seen, combined = set(), []
        for p in img_slot:
            if p not in seen:
                seen.add(p)
                combined.append(p)
        for p in txt_slot:
            if p not in seen:
                seen.add(p)
                combined.append(p)
        return combined  # up to 2*top_k items


def build_relevant_set(item: dict) -> tuple:
    """Build source-aware relevant set and primary item from a QA entry.

    Returns:
        (relevant_set, primary_item) where both use (doc_basename, page) tuples.
    """
    docs  = item["relevant_docs"]
    pages = item["relevant_pages"]
    primary_page = item["primary_page"]

    if not docs:
        return set(), None

    primary_doc = docs[0]

    if len(docs) == 1:
        relset = {(docs[0], p) for p in pages}
    else:
        relset = {(docs[i], pages[i]) for i in range(len(pages))}

    return relset, (primary_doc, primary_page)


# ── Metric functions ──────────────────────────────────────────────────────────

def hit_at_k(retrieved: list, relevant: set, k: int) -> int:
    return int(any(p in relevant for p in retrieved[:k]))


def hit(retrieved: list, relevant: set) -> int:
    """Hit over the full retrieved context (variable size for dual-channel methods)."""
    return int(any(p in relevant for p in retrieved))


def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
    if k == 0:
        return 0.0
    return sum(1 for p in retrieved[:k] if p in relevant) / k


def recall_at_k(retrieved: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    return sum(1 for p in retrieved[:k] if p in relevant) / len(relevant)


def recall(retrieved: list, relevant: set) -> float:
    """Recall over the full retrieved context."""
    if not relevant:
        return 0.0
    return sum(1 for p in retrieved if p in relevant) / len(relevant)


def mrr(retrieved: list, relevant: set) -> float:
    """MRR over the full retrieved list."""
    for rank, p in enumerate(retrieved, 1):
        if p in relevant:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved: list, relevant_set: set, primary_item: tuple, k: int) -> float:
    """NDCG@K with graded relevance: primary item = 2, other relevant = 1, else 0."""
    def rel(p):
        if primary_item is not None and p == primary_item:  return 2
        if p in relevant_set:  return 1
        return 0

    top  = retrieved[:k]
    dcg  = sum(rel(p) / math.log2(rank + 1) for rank, p in enumerate(top, 1))
    ideal_items = (relevant_set | {primary_item}) if primary_item is not None else relevant_set
    ideal_grades = sorted([rel(p) for p in ideal_items], reverse=True)
    idcg = sum(g / math.log2(rank + 1) for rank, g in enumerate(ideal_grades[:k], 1))
    return dcg / idcg if idcg > 0 else 0.0


print("Retrieval metric functions defined (source-aware, full-context for dual-channel methods).")

In [ ]:
retrieval_rows = []

for item in QA_ITEMS:
    qid   = item["id"]
    qtype = item["question_type"]
    diff  = item["difficulty"]

    if not item.get("relevant_docs"):  # out_of_scope: no retrieval ground truth
        continue

    relevant_set, primary_item = build_relevant_set(item)

    for m in METHODS:
        result = ALL_RESULTS.get((m["name"], qid))
        if result is None:
            retrieval_rows.append({
                "method": m["label"], "question_id": qid,
                "question_type": qtype, "difficulty": diff,
                "ctx_size": float("nan"),
                "hit": float("nan"), "recall": float("nan"),
                "p_k": float("nan"), "r_k": float("nan"),
                "mrr": float("nan"), "ndcg_k": float("nan"), "ndcg_ctx": float("nan"),
            })
            continue

        pages    = get_retrieved_pages(result, m["name"])
        ctx_size = len(pages)
        hits_in  = sum(1 for p in pages if p in relevant_set)

        row = {
            "method":        m["label"],
            "question_id":   qid,
            "question_type": qtype,
            "difficulty":    diff,
            "ctx_size":      ctx_size,
            # Full-context metrics (dual-channel methods evaluated over up to 2*TOP_K items)
            "hit":     hit(pages, relevant_set),
            "recall":  recall(pages, relevant_set),
            "mrr":     mrr(pages, relevant_set),
            "ndcg_ctx": ndcg_at_k(pages, relevant_set, primary_item, ctx_size),
            # Fixed-K metrics for fair single-channel comparison
            "hit_k":   hit_at_k(pages, relevant_set, TOP_K),
            "p_k":     precision_at_k(pages, relevant_set, TOP_K),
            "r_k":     recall_at_k(pages, relevant_set, TOP_K),
            "ndcg_k":  ndcg_at_k(pages, relevant_set, primary_item, TOP_K),
        }
        retrieval_rows.append(row)

df_retrieval = pd.DataFrame(retrieval_rows)
print(f"Retrieval metrics computed: {len(df_retrieval)} rows (source-aware, full-context)")
display(df_retrieval.head(14))

## 6. Answer Quality Metrics

In [ ]:
def normalize_answer(s: str) -> str:
    """Lowercase, remove punctuation, collapse whitespace."""
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def exact_match(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(gold_tokens)
    n_common = sum(common.values())
    if n_common == 0:
        return 0.0
    p = n_common / len(pred_tokens)
    r = n_common / len(gold_tokens)
    return 2 * p * r / (p + r)


def rouge_n(pred: str, gold: str, n: int) -> float:
    """ROUGE-N recall."""
    def ngrams(tokens, n):
        return Counter(zip(*[tokens[i:] for i in range(n)]))
    pred_toks = normalize_answer(pred).split()
    gold_toks = normalize_answer(gold).split()
    gold_ng = ngrams(gold_toks, n)
    pred_ng = ngrams(pred_toks, n)
    if not gold_ng:
        return 0.0
    overlap = sum((gold_ng & pred_ng).values())
    return overlap / sum(gold_ng.values())


def rouge_l(pred: str, gold: str) -> float:
    """ROUGE-L based on LCS length."""
    pred_toks = normalize_answer(pred).split()
    gold_toks = normalize_answer(gold).split()
    if not pred_toks or not gold_toks:
        return 0.0
    m, n = len(gold_toks), len(pred_toks)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if gold_toks[i-1] == pred_toks[j-1] \
                        else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    p_ = lcs / n
    r_ = lcs / m
    return 2 * p_ * r_ / (p_ + r_) if (p_ + r_) > 0 else 0.0


def contains_answer(pred: str, gold: str) -> int:
    """1 if normalized gold answer appears as substring in normalized pred."""
    return int(normalize_answer(gold) in normalize_answer(pred))


print("Answer quality metric functions defined.")


In [ ]:
# Semantic similarity via sentence-transformers (all-MiniLM-L6-v2, runs on CPU)
from sentence_transformers import SentenceTransformer, util as st_util

_sem_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

def semantic_sim(pred: str, gold: str) -> float:
    """Cosine similarity of MiniLM embeddings — range [0, 1]."""
    if not pred.strip() or not gold.strip():
        return 0.0
    embs = _sem_model.encode([pred, gold], convert_to_tensor=True, show_progress_bar=False)
    return float(st_util.cos_sim(embs[0], embs[1]).item())

print("Semantic similarity function defined (all-MiniLM-L6-v2 on CPU).")


In [ ]:
# Only skip out_of_scope — visual_only questions CAN be scored
# (their gold answers are short factual strings from figure captions)
SKIP_TYPES = {"out_of_scope"}

answer_rows = []

for item in QA_ITEMS:
    qid   = item["id"]
    gold  = item["answer"]
    qtype = item["question_type"]
    diff  = item["difficulty"]

    for m in METHODS:
        result = ALL_RESULTS.get((m["name"], qid))

        if result is None or qtype in SKIP_TYPES:
            answer_rows.append({
                "method": m["label"], "question_id": qid,
                "question_type": qtype, "difficulty": diff,
                "em": float("nan"), "token_f1": float("nan"),
                "rouge1": float("nan"), "rouge2": float("nan"),
                "rougeL": float("nan"), "contains": float("nan"),
                "sem_sim": float("nan"),
            })
            continue

        pred = result["answer"]
        answer_rows.append({
            "method":        m["label"],
            "question_id":   qid,
            "question_type": qtype,
            "difficulty":    diff,
            "em":       exact_match(pred, gold),
            "token_f1": token_f1(pred, gold),
            "rouge1":   rouge_n(pred, gold, 1),
            "rouge2":   rouge_n(pred, gold, 2),
            "rougeL":   rouge_l(pred, gold),
            "contains": contains_answer(pred, gold),
            "sem_sim":  semantic_sim(pred, gold),
        })

df_answer = pd.DataFrame(answer_rows)
print(f"Answer quality metrics computed: {len(df_answer)} rows")
display(df_answer.head(7))


## 7. Summary Tables

### Table 1 — Overall Retrieval Metrics by Method

In [ ]:
metric_cols = ["hit", "recall", "mrr", "ndcg_ctx", "hit_k", "p_k", "r_k", "ndcg_k"]
col_rename  = {
    "hit": "Hit@ctx", "recall": "Recall@ctx", "mrr": "MRR",
    "ndcg_ctx": "NDCG@ctx",
    "hit_k": f"Hit@{TOP_K}", "p_k": f"P@{TOP_K}",
    "r_k": f"R@{TOP_K}", "ndcg_k": f"NDCG@{TOP_K}",
}

df_ret_summary = (
    df_retrieval
    .groupby("method")[metric_cols + ["ctx_size"]]
    .mean()
    .rename(columns={**col_rename, "ctx_size": "avg_ctx"})
    .sort_values("NDCG@ctx", ascending=False)
    .round(4)
)

# Reorder rows to match METHODS order
display(
    df_ret_summary.style
    .highlight_max(axis=0, color="lightgreen")
    .format("{:.4f}")
    .set_caption(
        f"Table 1 — Overall Retrieval Metrics (n=115 questions)\n"
        f"@ctx = full LLM context: {TOP_K} items for single-channel, up to 2×{TOP_K} for dual-channel\n"
        f"@{TOP_K} = fixed-K comparison across all methods"
    )
)

### Table 2 — Retrieval Hit@K by Question Type and Difficulty

In [ ]:
diff_order = ["easy", "medium", "hard"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, diff in zip(axes, diff_order):
    sub = df_retrieval[df_retrieval["difficulty"] == diff]
    if sub.empty:
        ax.set_title(f"{diff.capitalize()} (no data)")
        ax.axis("off")
        continue

    pivot = sub.pivot_table(
        index="question_type", columns="method",
        values="hit_k", aggfunc="mean"
    ).reindex(columns=METHOD_LABELS, fill_value=float("nan"))

    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(METHOD_LABELS)))
    ax.set_xticklabels(METHOD_LABELS, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_title(f"{diff.capitalize()} questions", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for i in range(len(pivot.index)):
        for j in range(len(METHOD_LABELS)):
            val = pivot.values[i, j]
            if not math.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=7, color="black")

plt.suptitle(f"Table 2 — Hit@{TOP_K} by Question Type x Method", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


### Table 3 — Answer Quality by Method

In [ ]:
ans_cols   = ["em", "token_f1", "rouge1", "rouge2", "rougeL", "contains", "sem_sim"]
ans_rename = {"em": "EM", "token_f1": "Token-F1", "rouge1": "ROUGE-1",
              "rouge2": "ROUGE-2", "rougeL": "ROUGE-L", "contains": "Contains",
              "sem_sim": "Sem-Sim"}

df_ans_summary = (
    df_answer
    .groupby("method")[ans_cols]
    .mean()
    .rename(columns=ans_rename)
    .sort_values("Sem-Sim", ascending=False)
    .round(4)
)

display(
    df_ans_summary.style
    .highlight_max(axis=0, color="lightgreen")
    .format("{:.4f}")
    .set_caption("Table 3 — Answer Quality by Method (mean over scoreable questions)")
)


### Table 4 — Ablation: hindex Overlay Contribution

Each hindex-overlay method paired with its base method to isolate the contribution of the hierarchical index.


In [ ]:
ABLATION_PAIRS = [
    ("Embedding",   "Embed+hindex",   "+hindex"),
    ("Text Search", "Text+hindex",    "+hindex"),
    ("Hybrid",      "Hybrid+hindex",  "+hindex"),
]

ablation_rows = []
for base_label, overlay_label, tag in ABLATION_PAIRS:
    for metric in [f"Hit@{TOP_K}", f"P@{TOP_K}", f"R@{TOP_K}", "MRR", f"NDCG@{TOP_K}"]:
        orig_col = {"Hit@4": "hit_k", "P@4": "p_k", "R@4": "r_k",
                    "MRR": "mrr", "NDCG@4": "ndcg_k"}[metric]
        base_val    = df_retrieval[df_retrieval["method"] == base_label   ][orig_col].mean()
        overlay_val = df_retrieval[df_retrieval["method"] == overlay_label][orig_col].mean()
        delta = overlay_val - base_val
        ablation_rows.append({
            "Base method": base_label, "Overlay": overlay_label,
            "Metric": metric,
            "Base": round(base_val, 4), "Overlay": round(overlay_val, 4),
            "Δ": f"{delta:+.4f}",
        })

df_ablation = pd.DataFrame(ablation_rows)

# Pivot for compact view
ablation_pivot = df_ablation.pivot_table(
    index=["Base method", "Overlay"],
    columns="Metric", values=["Base", "Overlay", "Δ"],
    aggfunc="first"
)

display(
    df_ablation.pivot(
        index=["Base method", "Overlay"], columns="Metric", values="Δ"
    ).style
    .set_caption(f"Table 4 — Δ (overlay − base) from adding hindex (positive = improvement)")
    .format("{}")
    .map(lambda v: "color: green" if isinstance(v, str) and v.startswith("+") else "color: red")
)


### Table 5 — Statistical Significance (Wilcoxon Signed-Rank Test)

Pairwise one-sided test on per-question NDCG@4: *row method is better than column method*.  
\* p < 0.05 · \*\* p < 0.01 · \*\*\* p < 0.001


In [ ]:
from scipy import stats as scipy_stats

METHOD_ORDER = [m["label"] for m in METHODS]
ndcg_by_method = {
    label: df_retrieval[df_retrieval["method"] == label]["ndcg_k"].values
    for label in METHOD_ORDER
}

n_m = len(METHOD_ORDER)
pval_matrix = np.ones((n_m, n_m))
# Effect size: rank-biserial correlation r = W / (n*(n+1)/2)
effect_matrix = np.zeros((n_m, n_m))

for i, m1 in enumerate(METHOD_ORDER):
    for j, m2 in enumerate(METHOD_ORDER):
        if i == j:
            continue
        v1, v2 = ndcg_by_method[m1], ndcg_by_method[m2]
        mask = ~(np.isnan(v1) | np.isnan(v2))
        n_pairs = mask.sum()
        if n_pairs < 10:
            continue
        diff = v1[mask] - v2[mask]
        if (diff == 0).all():
            continue
        try:
            stat, pval = scipy_stats.wilcoxon(diff, alternative="greater", zero_method="wilcox")
            pval_matrix[i, j] = pval
            # Rank-biserial correlation as effect size
            effect_matrix[i, j] = 1 - (2 * stat) / (n_pairs * (n_pairs + 1))
        except Exception:
            pass


def _sig(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""


sig_table = pd.DataFrame(
    [[f"{pval_matrix[i,j]:.3f}{_sig(pval_matrix[i,j])}" if i != j else "—"
      for j in range(n_m)]
     for i in range(n_m)],
    index=METHOD_ORDER, columns=METHOD_ORDER,
)

effect_table = pd.DataFrame(
    [[f"{effect_matrix[i,j]:.3f}" if i != j else "—"
      for j in range(n_m)]
     for i in range(n_m)],
    index=METHOD_ORDER, columns=METHOD_ORDER,
)

print("Table 5a — p-values (row beats col on NDCG@4, one-sided Wilcoxon):")
display(sig_table.style.set_caption("Table 5a — p-values (* p<0.05, ** p<0.01, *** p<0.001)"))
print()
print("Table 5b — Rank-biserial correlation (effect size, range −1 to +1):")
display(effect_table.style.set_caption("Table 5b — Effect size (rank-biserial r)"))


### Table 6 — Bootstrap 95 % Confidence Intervals (NDCG@4)

Per-question NDCG@4 values resampled 10 000 times (with replacement) to estimate the 95 % CI for each method's mean.  
Non-overlapping CIs indicate statistically distinguishable performance.


In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 10_000

ci_rows = []
boot_distributions = {}

for m in METHODS:
    scores = df_retrieval[df_retrieval["method"] == m["label"]]["ndcg_k"].dropna().values
    if len(scores) == 0:
        continue
    # Vectorised bootstrap
    idx = rng.integers(0, len(scores), size=(N_BOOT, len(scores)))
    boot_means = scores[idx].mean(axis=1)
    boot_distributions[m["label"]] = boot_means
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    ci_rows.append({
        "Method":          m["label"],
        f"NDCG@{TOP_K}":   round(float(scores.mean()), 4),
        "95% CI [low":     round(ci_low, 4),
        "high]":           round(ci_high, 4),
        "CI width":        round(ci_high - ci_low, 4),
        "n":               len(scores),
    })

df_ci = pd.DataFrame(ci_rows).sort_values(f"NDCG@{TOP_K}", ascending=False)
display(df_ci.style
    .set_caption(f"Table 6 — Bootstrap 95 % CI for NDCG@{TOP_K} ({N_BOOT:,} resamples, n={len(scores)} questions)")
    .format({f"NDCG@{TOP_K}": "{:.4f}", "95% CI [low": "{:.4f}", "high]": "{:.4f}", "CI width": "{:.4f}", "n": "{:d}"})
    .highlight_max(subset=[f"NDCG@{TOP_K}"], color="lightgreen")
)

# Visual: CI forest plot
fig, ax = plt.subplots(figsize=(9, 5))
for i, row in df_ci.reset_index(drop=True).iterrows():
    label = row["Method"]
    color = next(m["color"] for m in METHODS if m["label"] == label)
    mean  = row[f"NDCG@{TOP_K}"]
    lo    = row["95% CI [low"]
    hi    = row["high]"]
    y     = len(df_ci) - 1 - i
    ax.plot([lo, hi], [y, y], color=color, linewidth=3)
    ax.plot(mean, y, "o", color=color, markersize=8, label=label)
ax.set_yticks(range(len(df_ci)))
ax.set_yticklabels(df_ci["Method"].tolist()[::-1], fontsize=9)
ax.set_xlabel(f"NDCG@{TOP_K}")
ax.set_title(f"Figure 5 — Bootstrap 95 % CI Forest Plot for NDCG@{TOP_K}", fontsize=11)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Visualizations

### Figure 1 — Radar Chart (5 retrieval metrics, all 7 methods)

In [ ]:
radar_metrics = [f"Hit@{TOP_K}", f"P@{TOP_K}", f"R@{TOP_K}", "MRR", f"NDCG@{TOP_K}"]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=10)

for m in METHODS:
    label = m["label"]
    if label not in df_ret_summary.index:
        continue
    vals = [df_ret_summary.loc[label, col] for col in radar_metrics]
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, label=label, color=m["color"])
    ax.fill(angles, vals, alpha=0.08, color=m["color"])

ax.set_ylim(0, 1)
ax.set_title(f"Figure 1 — Retrieval Metrics Radar (Top-{TOP_K})", pad=20, fontsize=12)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()


### Figure 2 — Grouped Bar Chart (Hit@K, P@K, R@K)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

bar_metrics = [f"Hit@{TOP_K}", f"P@{TOP_K}", f"R@{TOP_K}"]
n_methods   = len(METHODS)
n_groups    = len(bar_metrics)
bar_width   = 0.10
group_gap   = 0.3

group_centers = np.arange(n_groups) * (n_methods * bar_width + group_gap)

for i, m in enumerate(METHODS):
    label = m["label"]
    vals  = []
    for col in bar_metrics:
        v = df_ret_summary.loc[label, col] if label in df_ret_summary.index else float("nan")
        vals.append(v if not math.isnan(v) else 0)
    x_pos = group_centers + (i - n_methods / 2 + 0.5) * bar_width
    ax.bar(x_pos, vals, width=bar_width, label=label, color=m["color"], alpha=0.85)

# Horizontal line at best value per group
for g_idx, col in enumerate(bar_metrics):
    best = df_ret_summary[col].max()
    x0 = group_centers[g_idx] - n_methods * bar_width / 2
    x1 = group_centers[g_idx] + n_methods * bar_width / 2
    ax.hlines(best, x0, x1, colors="black", linestyles="--", linewidth=1.0)

ax.set_xticks(group_centers)
ax.set_xticklabels(bar_metrics, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title(f"Figure 2 — Grouped Bar: Hit@{TOP_K}, P@{TOP_K}, R@{TOP_K}", fontsize=12)
ax.legend(fontsize=8, ncol=2, loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Figure 3 — Hit@K by Question Type (side-by-side bars)

In [ ]:
qtypes = sorted(df_retrieval["question_type"].unique())
n_qtypes  = len(qtypes)
n_methods = len(METHODS)
bar_width = 0.10
group_gap = 0.2

fig, ax = plt.subplots(figsize=(16, 6))
group_centers = np.arange(n_qtypes) * (n_methods * bar_width + group_gap)

for i, m in enumerate(METHODS):
    label = m["label"]
    sub   = df_retrieval[df_retrieval["method"] == label]
    vals  = [sub[sub["question_type"] == qt]["hit_k"].mean() for qt in qtypes]
    vals  = [v if not math.isnan(v) else 0 for v in vals]
    x_pos = group_centers + (i - n_methods / 2 + 0.5) * bar_width
    ax.bar(x_pos, vals, width=bar_width, label=label, color=m["color"], alpha=0.85)

ax.set_xticks(group_centers)
ax.set_xticklabels(qtypes, rotation=25, ha="right", fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel(f"Hit@{TOP_K}")
ax.set_title(f"Figure 3 — Hit@{TOP_K} by Question Type", fontsize=12)
ax.legend(fontsize=8, ncol=2, loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Figure 4 — Answer Quality (horizontal bars, 2x2 grid)

In [ ]:
ans_metrics = ["Token-F1", "ROUGE-1", "ROUGE-L", "Sem-Sim"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, metric in zip(axes, ans_metrics):
    vals = []
    for m in METHODS:
        label = m["label"]
        v = df_ans_summary.loc[label, metric] if label in df_ans_summary.index else float("nan")
        vals.append((label, v if not math.isnan(v) else 0, m["color"]))
    vals.sort(key=lambda x: x[1])

    labels_ = [v[0] for v in vals]
    scores_ = [v[1] for v in vals]
    colors_ = [v[2] for v in vals]

    bars = ax.barh(labels_, scores_, color=colors_, alpha=0.85)
    ax.set_xlim(0, max(scores_) * 1.15 + 0.01)
    ax.set_xlabel("Score")
    ax.set_title(metric, fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    for bar, score in zip(bars, scores_):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{score:.3f}", va="center", fontsize=8)

plt.suptitle("Figure 4 — Answer Quality Comparison by Method", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 9. Per-Question Deep Dive

In [ ]:
DIFF_ORDER = {"easy": 0, "medium": 1, "hard": 2}
sorted_items = sorted(QA_ITEMS, key=lambda x: (DIFF_ORDER.get(x["difficulty"], 9), x["question_type"]))

for item in sorted_items:
    qid    = item["id"]
    qtype  = item["question_type"]
    diff   = item["difficulty"]
    q      = item["question"]
    gold   = item["answer"]

    if item.get("relevant_docs"):
        relevant_set, primary_item = build_relevant_set(item)
    else:
        relevant_set, primary_item = set(), None  # out_of_scope: no retrieval ground truth

    # Compact page display: for cross-doc, abbreviate doc name; otherwise just page number.
    is_cross = (len(item["relevant_docs"]) > 1
                or len(item["relevant_pages"]) > len(item["relevant_docs"]))

    def fmt_page(src_page):
        src, p = src_page
        if is_cross and src:
            parts = src.replace(".pdf", "").split("-")
            abbr = parts[-1][:4] if parts else src[:4]
            return f"{abbr}:p{p}"
        return f"p{p}"

    print(f"[{qid}] {qtype}/{diff}  \"{q[:60]}\"")
    for m in METHODS:
        result = ALL_RESULTS.get((m["name"], qid))
        if result is None:
            print(f"  {m['label']:<14} ERROR (no result)")
            continue

        pages = get_retrieved_pages(result, m["name"])
        hit   = hit_at_k(pages, relevant_set, TOP_K)
        lat   = result["latency_s"]
        pred  = result["answer"]

        f1_str = "N/A " if qtype in SKIP_TYPES else f"{token_f1(pred, gold):.2f}"

        if hit:
            hit_pages = [p for p in pages[:TOP_K] if p in relevant_set]
            hit_str = "✓ " + " ".join(fmt_page(p) for p in hit_pages)
        else:
            hit_str = "✗"

        page_str = ",".join(fmt_page(p) for p in pages[:TOP_K])
        print(f"  {m['label']:<14} {hit_str:<20} pages=[{page_str}]  F1={f1_str}  {lat:.1f}s")
    print()


## 10. Key Findings

**Best overall retrieval:** {method} (NDCG@4 = {score})

**Best for visual-only questions:** {method}

**Best for exact lookup:** {method}

**Best for cross-document questions:** {method}

**Fastest method:** {method} (avg {X}s)

**Hallucination resistance (out-of-scope):** methods that correctly declined vs. those that fabricated

### Trade-off matrix

| Method | Retrieval | Answer Quality | Speed | Best use-case |
|--------|-----------|----------------|-------|---------------|
| Embedding | {val} | {val} | {val} | Semantic / paraphrase queries |
| Text Search | {val} | {val} | {val} | Exact keyword lookup |
| Hybrid | {val} | {val} | {val} | General-purpose baseline |
| hindex only | {val} | {val} | {val} | Structured document navigation |
| Embed+hindex | {val} | {val} | {val} | Semantic + structure |
| Text+hindex | {val} | {val} | {val} | Keyword + structure |
| Hybrid+hindex | {val} | {val} | {val} | Maximum recall |

*Run the cell below to compute actual values for the placeholders above.*


In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

def _best(series, label=""):
    s = series.dropna()
    if s.empty:
        return "N/A", float("nan")
    return s.idxmax(), s.max()

# Best overall retrieval
best_ret_m, best_ret_v = _best(df_ret_summary[f"NDCG@{TOP_K}"])
print(f"\nBest overall retrieval:          {best_ret_m} (NDCG@{TOP_K} = {best_ret_v:.4f})")

# Best by question type
for qt in ["visual_only", "exact_lookup", "table_cell", "cross_doc"]:
    sub = df_retrieval[df_retrieval["question_type"] == qt]
    best_m, best_v = _best(sub.groupby("method")[f"ndcg_k"].mean(), qt)
    print(f"Best NDCG@{TOP_K} for {qt:<22} {best_m} ({best_v:.4f})")

# Best answer quality
best_ans_m, best_ans_v = _best(df_ans_summary["Sem-Sim"])
print(f"\nBest answer quality (Sem-Sim):   {best_ans_m} ({best_ans_v:.4f})")

# Latency
lat_by_method = {}
for m in METHODS:
    lats = [r["latency_s"] for (mn, _), r in ALL_RESULTS.items() if mn == m["name"] and "latency_s" in r]
    if lats:
        lat_by_method[m["label"]] = np.mean(lats)

if lat_by_method:
    fastest = min(lat_by_method, key=lat_by_method.get)
    print(f"Fastest method:                  {fastest} (avg {lat_by_method[fastest]:.1f}s)")
    print(f"\nLatency summary (avg s/query):")
    for label, lat in sorted(lat_by_method.items(), key=lambda x: x[1]):
        print(f"  {label:<16} {lat:.2f}s")

# Significance summary — methods that significantly beat Embedding
if "ndcg_by_method" in dir():
    emb_idx = METHOD_ORDER.index("Embedding")
    print(f"\nMethods that significantly beat Embedding (NDCG@{TOP_K}, p<0.05):")
    for i, label in enumerate(METHOD_ORDER):
        if i != emb_idx and pval_matrix[i, emb_idx] < 0.05:
            p = pval_matrix[i, emb_idx]
            r = effect_matrix[i, emb_idx]
            stars = _sig(p)
            print(f"  {label:<16} p={p:.4f}{stars}  r={r:.3f}")

# Hallucination resistance
print(f"\nOut-of-scope question handling:")
oos_items = [it for it in QA_ITEMS if it["question_type"] == "out_of_scope"]
refusals = {m["label"]: 0 for m in METHODS}
for item in oos_items:
    for m in METHODS:
        r = ALL_RESULTS.get((m["name"], item["id"]))
        if r:
            ans = (r["answer"] or "").lower()
            if any(kw in ans for kw in ("not mentioned", "not provided", "not available",
                                         "no information", "cannot", "not specified",
                                         "not explicitly", "not stated")):
                refusals[m["label"]] += 1
for label, cnt in sorted(refusals.items(), key=lambda x: -x[1]):
    print(f"  {label:<16} refused {cnt}/{len(oos_items)} out-of-scope questions")

# Bootstrap CI summary
if "df_ci" in dir():
    print(f"\nConfidence intervals overlap analysis (NDCG@{TOP_K}):")
    best_lo = df_ci.iloc[0]["95% CI [low"]
    worse_methods = [row["Method"] for _, row in df_ci.iterrows() if row["high]"] < best_lo]
    if worse_methods:
        print(f"  Methods with CI entirely below {df_ci.iloc[0]['Method']}: {worse_methods}")
    else:
        print(f"  All method CIs overlap with {df_ci.iloc[0]['Method']} (no clear separation at 95% level)")


In [ ]:
EXPORT_DIR = Path(app_dir)

exports = {
    "eval_retrieval_metrics.csv":  (df_retrieval, False),
    "eval_answer_metrics.csv":     (df_answer,    False),
    "eval_retrieval_summary.csv":  (df_ret_summary, True),
    "eval_answer_summary.csv":     (df_ans_summary, True),
    "eval_ablation.csv":           (df_ablation,  False),
    "eval_bootstrap_ci.csv":       (df_ci,        False),
}

for fname, (df, with_index) in exports.items():
    path = EXPORT_DIR / fname
    df.to_csv(path, index=with_index)
    print(f"  {fname:<38} ({len(df)} rows)")

# Significance tables
sig_table.to_csv(EXPORT_DIR / "eval_significance_pvalues.csv")
effect_table.to_csv(EXPORT_DIR / "eval_significance_effects.csv")
print(f"  {'eval_significance_pvalues.csv':<38} ({sig_table.shape[0]}x{sig_table.shape[1]})")
print(f"  {'eval_significance_effects.csv':<38} ({effect_table.shape[0]}x{effect_table.shape[1]})")

print(f"\nAll exports written to: {EXPORT_DIR}")
